# Stage 1 — Modeling | Reactor–Separator Process (2 CSTRs + flash tank)
First-principles model, steady-state and dynamic datasets, and AI models (Linear, MLP, TSK-Fuzzy), followed by Stage 2 (optimization) and Stage 3 (control) built on them.

## Import libraries

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve
np.set_printoptions(precision=4, suppress=True, linewidth=140)

import os
OUTPUT_DIR = "outputs"   # portable: works in Colab, Jupyter, or any local machine
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Process parameters
All constants of the model in one place, so they are easy to check and change. Units: time [h], temperature [K], heat input Q [kJ/h], volumes [m³], flows [m³/h].

In [ ]:
# ---------- Model parameters ----------
R      = 8.314            # kJ/(kmol K)
rho    = 1000.0           # kg/m3
cp     = 4.2              # kJ/(kg K)
V      = np.array([1.0, 0.5, 1.0])              # m3  (vessel 1, 2, 3)
alpha  = np.array([3.5, 1.0, 0.5])              # relative volatilities A, B, C
E1, E2 = 5.0e4, 6.0e4                            # kJ/kmol
dH1, dH2 = -6.0e4, -7.0e4                        # kJ/kmol (exothermic)
dHvap  = np.array([-3.53e4, -1.57e4, -4.068e4])  # kJ/kmol
T10 = T20 = 300.0                                # K   feed temperatures
xA10, xB10, xA20, xB20 = 1.0, 0.0, 1.0, 0.0      # pure A feeds
# Flows
F10, F20, Fr, Fp = 5.04, 5.04, 50.4, 0.504       # m3/h
F1 = F10 + Fr;  F2 = F1 + F20                    # m3/h (from flow balances)

# Pre-exponential factors: source lists 2.77e3 and 2.6e3 "s^-1".
# With the x3600 conversion the mass balances close (dx/dt ~ 0) at the published steady state.
k1 = 2.77e3 * 3600.0                             # h^-1
k2 = 2.60e3 * 3600.0                             # h^-1

# Energy-balance unit corrections (calibrated against the published steady state).
# The published parameters give a non-zero dT/dt at the published steady state unless the
# reaction-heat terms are scaled by C_RXN and the vaporization-heat term by C_VAP.
# Both factors came out as clean numbers (0.00400 from T1 and T2 independently; 2.0000 from T3).
C_RXN = 0.004
C_VAP = 2.0

# ---------- Published reference values ----------
DT_PAPER   = 0.005                                # h sampling period
U_MAX      = np.array([4.87e6, 1.68e6, 4.87e6])   # kJ/h (upper bound)
U_MIN      = np.zeros(3)                          # kJ/h (lower bound)
U_S        = np.array([2.90e6, 1.00e6, 2.90e6])   # kJ/h steady-state input
XS_PAPER_2DP = np.array([0.18,0.67,480.32,0.20,0.65,472.79,0.07,0.67,474.89])   #
XS_REF_4DP   = np.array([0.1763,0.6731,480.3165,0.1965,0.6536,472.7863,0.0651,0.6703,474.8877])  # same steady state, 4 dp
X_INIT_PAPER = np.array([0.1155,0.6235,497.3,0.1367,0.6053,489.8,0.0396,0.5504,491.8])          # data-generation initial state

STATE_NAMES = ["xA1","xB1","T1","xA2","xB2","T2","xA3","xB3","T3"]
INPUT_NAMES = ["Q1","Q2","Q3"]

## First-principles equations
The 9 ODEs (mass balances of A and B and an energy balance in each vessel) plus 3 algebraic equations for the flash overhead composition.

Inputs: state x (9) and heat inputs u = [Q1, Q2, Q3]. Output: dx/dt (9). The outputs are the states themselves (y = x, all states measured).

In [ ]:
def rhs(t, x, u):
    """dx/dt = f(x,u).  u: array [Q1,Q2,Q3] (kJ/h) or callable u(t)."""
    Q = u(t) if callable(u) else u
    xA1,xB1,T1, xA2,xB2,T2, xA3,xB3,T3 = x
    # --- algebraic: flash overhead composition (relative volatility) ---
    xC3 = 1.0 - xA3 - xB3
    den = alpha[0]*xA3 + alpha[1]*xB3 + alpha[2]*xC3
    xAr, xBr, xCr = alpha[0]*xA3/den, alpha[1]*xB3/den, alpha[2]*xC3/den
    # --- reaction rates (Arrhenius) ---
    a1, b1 = k1*np.exp(-E1/(R*T1)), k2*np.exp(-E2/(R*T1))   # vessel 1: A->B, B->C
    a2, b2 = k1*np.exp(-E1/(R*T2)), k2*np.exp(-E2/(R*T2))   # vessel 2
    # --- CSTR 1 ---
    dxA1 = F10/V[0]*(xA10-xA1) + Fr/V[0]*(xAr-xA1) - a1*xA1
    dxB1 = F10/V[0]*(xB10-xB1) + Fr/V[0]*(xBr-xB1) + a1*xA1 - b1*xB1
    dT1  = (F10/V[0]*(T10-T1) + Fr/V[0]*(T3-T1)
            - C_RXN*(dH1/cp*a1*xA1 + dH2/cp*b1*xB1) + Q[0]/(rho*cp*V[0]))
    # --- CSTR 2 ---
    dxA2 = F1/V[1]*(xA1-xA2) + F20/V[1]*(xA20-xA2) - a2*xA2
    dxB2 = F1/V[1]*(xB1-xB2) + F20/V[1]*(xB20-xB2) + a2*xA2 - b2*xB2
    dT2  = (F1/V[1]*(T1-T2) + F20/V[1]*(T20-T2)
            - C_RXN*(dH1/cp*a2*xA2 + dH2/cp*b2*xB2) + Q[1]/(rho*cp*V[1]))
    # --- flash separator ---
    dxA3 = F2/V[2]*(xA2-xA3) - (Fr+Fp)/V[2]*(xAr-xA3)
    dxB3 = F2/V[2]*(xB2-xB3) - (Fr+Fp)/V[2]*(xBr-xB3)
    dT3  = (F2/V[2]*(T2-T3) + Q[2]/(rho*cp*V[2])
            + C_VAP*(Fr+Fp)/(rho*cp*V[2])*(xAr*dHvap[0]+xBr*dHvap[1]+xCr*dHvap[2]))
    return np.array([dxA1,dxB1,dT1,dxA2,dxB2,dT2,dxA3,dxB3,dT3])

def is_physical(x):
    """Mass fractions in [0,1], xA+xB<=1 in each vessel, temperatures > 0 K and finite."""
    x = np.asarray(x, float)
    if not np.all(np.isfinite(x)): return False
    xa, xb, T = x[[0,3,6]], x[[1,4,7]], x[[2,5,8]]
    return bool(np.all(xa>=-1e-6) and np.all(xb>=-1e-6) and np.all(xa+xb<=1+1e-6) and np.all(T>0))

def simulate(u, x0, t_end, dt=DT_PAPER):
    """Integrate the ODEs. Returns dict(t, X[n,9], ok, msg). Detects solver failure and non-physical states."""
    t_eval = np.arange(0.0, t_end + dt/2, dt)
    sol = solve_ivp(rhs, (0.0, t_end), np.asarray(x0, float), args=(u,), method="LSODA",
                    t_eval=t_eval, rtol=1e-8, atol=1e-10, max_step=dt)
    X = sol.y.T
    ok = bool(sol.success and X.shape[0]==len(t_eval) and all(is_physical(r) for r in X))
    return dict(t=sol.t, X=X, ok=ok, msg=sol.message)

def steady_state(u, x0=None, t_max=60.0, tol=1e-3):
    """Steady state for constant u: integrate to t_max, polish with fsolve, verify max|dx/dt|<tol."""
    x0 = XS_REF_4DP if x0 is None else x0
    sol = solve_ivp(rhs, (0.0, t_max), np.asarray(x0, float), args=(u,), method="LSODA", rtol=1e-8, atol=1e-10)
    if not sol.success: return None, dict(ok=False, reason=sol.message, resid=np.nan)
    x = fsolve(lambda z: rhs(0.0, z, u), sol.y[:, -1], xtol=1e-12)
    resid = float(np.max(np.abs(rhs(0.0, x, u))))
    ok = (resid < tol) and is_physical(x)
    return (x if ok else None), dict(ok=ok, reason="" if ok else "residual/physical check failed", resid=resid)

def jacobian_eigs(x, u, h=1e-6):
    J = np.zeros((9, 9))
    for i in range(9):
        d = np.zeros(9); d[i] = h*max(1.0, abs(x[i]))
        J[:, i] = (rhs(0, x+d, u) - rhs(0, x-d, u)) / (2*d[i])
    return np.linalg.eigvals(J)

## Test the process model
**Test A:** does the model reproduce the steady state published for u_s = [2.90, 1.00, 2.90]×10⁶ kJ/h?
**Test B:** transient from the paper's initial state under constant u_s. **Test C:** step change in Q1. **Test D:** local stability.

In [ ]:
# ---- Test A: steady state at u_s vs published values ----
xss, info = steady_state(U_S, x0=X_INIT_PAPER)          # start from the paper's (different) initial state
print("Steady-state solve OK:", info["ok"], "| max|dx/dt| =", f"{info['resid']:.2e}")
cmp = pd.DataFrame({"state": STATE_NAMES, "model": xss, "published_4dp": XS_REF_4DP,
                    "published_2dp": XS_PAPER_2DP})
cmp["abs_err_vs_4dp"] = (cmp["model"] - cmp["published_4dp"]).abs()
print(cmp.to_string(index=False))
tolA = np.r_[0.0006, 0.0006, 0.02]                       # rounding-level tolerance per (xA, xB, T)
passA = bool(np.all(cmp["abs_err_vs_4dp"].values <= np.tile(tolA, 3)))
print("TEST A (reproduces published steady state):", "PASS" if passA else "FAIL")

# ---- Test B: transient from paper's initial state, constant u_s ----
resB = simulate(U_S, X_INIT_PAPER, t_end=2.0)
print("TEST B solver ok / physical:", resB["ok"], "|", resB["msg"])

# ---- Test C: +10% step in Q1 at t = 1 h ----
def u_step(t):
    u = U_S.copy()
    if t >= 1.0: u[0] *= 1.10
    return u
resC = simulate(u_step, xss, t_end=3.0)
print("TEST C solver ok / physical:", resC["ok"])

# ---- Test D: local stability (open loop) ----
ev = jacobian_eigs(xss, U_S)
print("TEST D max Re(eig) =", f"{ev.real.max():.3f}", "->", "stable" if ev.real.max() < 0 else "UNSTABLE")

# ---- Plots ----
fig, ax = plt.subplots(3, 3, figsize=(13, 8), sharex=False)
for i, name in enumerate(STATE_NAMES):
    a = ax[i % 3][i // 3]
    a.plot(resB["t"], resB["X"][:, i], label="B: from x_init, u_s")
    a.plot(resC["t"], resC["X"][:, i], "--", label="C: +10% Q1 at 1 h")
    a.axhline(XS_REF_4DP[i], color="k", lw=0.6, ls=":")
    a.set_title(name); a.set_xlabel("time [h]")
ax[0][0].legend(fontsize=7); plt.tight_layout(); plt.show()
print("ALL TESTS PASSED" if (passA and resB["ok"] and resC["ok"] and ev.real.max() < 0) else "CHECK FAILURES ABOVE")

---
# Part B — Steady-state dataset, preprocessing, and AI models

The models follow the course's own worked example (ethylene-glycol reactive distillation): linear regression vs. a nonlinear MLP vs. a TSK (Sugeno) fuzzy model, fitted to simulated steady-state and dynamic data. Models are scored with RMSE, MAE and R².

Dynamic modeling uses the same three methods in a NARX formulation ("output is a function of past inputs and/or outputs") rather than an LSTM. MLPRegressor (scikit-learn) provides the MLP (hidden layers, ReLU, Adam, batch size, epochs), so no deep-learning framework is needed.

## Generate the steady-state dataset
`steady_state(u)` is solved at many points u = [Q1, Q2, Q3] spread over the input range using Latin Hypercube Sampling (LHS). With only 3 inputs, LHS covers every one-dimensional projection evenly with far fewer samples than a full grid would need for similar coverage.

**Input bounds:** the full bounds (0 to [4.87, 1.68, 4.87]×10⁶) give physically valid but extreme steady states (T1 down to ~233 K). The lower bound is therefore narrowed to [0.65, 0.22, 0.65]×10⁶ kJ/h to avoid that unrealistic cold corner. Change `U_LO` below to use the full bounds instead.

In [ ]:
# ---------- steady-state dataset ----------
from scipy.stats import qmc
np.random.seed(42)

U_LO = np.array([0.65e6, 0.22e6, 0.65e6])   # narrowed lower bound, see markdown above
U_HI = U_MAX.copy()                          # upper bound
N_SAMPLES = 2000                             # target size (Rule: PART 5)

sampler = qmc.LatinHypercube(d=3, seed=42)
U_unit = sampler.random(N_SAMPLES)
U_samples = U_LO + U_unit * (U_HI - U_LO)

rows = []
x_prev = XS_REF_4DP.copy()          # warm-start the solver from the last converged point
for i, u in enumerate(U_samples):
    xs, info = steady_state(u, x0=x_prev)
    rec = dict(sample_id=i, Q1=u[0], Q2=u[1], Q3=u[2], converged=info["ok"], residual=info["resid"])
    if xs is not None:
        for name, val in zip(STATE_NAMES, xs): rec[name] = val
        x_prev = xs                  # only warm-start from a converged point
    else:
        for name in STATE_NAMES: rec[name] = np.nan
    rows.append(rec)

df_ss = pd.DataFrame(rows)
n_ok = df_ss["converged"].sum()
print(f"Generated {len(df_ss)} samples | converged: {n_ok} | failed: {len(df_ss)-n_ok}")
df_ss.to_csv("steady_state_dataset.csv", index=False)
df_ss.head()

## Validate the steady-state dataset
Check for NaN/solver failures, duplicate inputs, and physically impossible mass fractions (should already be filtered by `is_physical` inside `steady_state`, but re-checked here for completeness). Also inspect the output ranges achieved.

In [ ]:
# ---------- validate ----------
clean = df_ss[df_ss["converged"]].copy().reset_index(drop=True)
n_failed = (~df_ss["converged"]).sum()
n_dup = clean.duplicated(subset=INPUT_NAMES).sum()
mass_bad = ((clean[["xA1","xB1"]].sum(axis=1) > 1 + 1e-6) |
            (clean[["xA2","xB2"]].sum(axis=1) > 1 + 1e-6) |
            (clean[["xA3","xB3"]].sum(axis=1) > 1 + 1e-6)).sum()

print("Solver failures (non-convergent / non-physical, dropped):", n_failed)
print("NaN in states:", df_ss[STATE_NAMES].isna().any(axis=1).sum())
print("Duplicate input rows:", n_dup)
print("Mass-fraction violations (xA+xB > 1) in the cleaned set:", mass_bad)
print(f"\nClean dataset size: {len(clean)}")
print(clean[STATE_NAMES].describe().T[["min","max","mean","std"]].round(3))

fig, axes = plt.subplots(3, 3, figsize=(12, 8))
for i, name in enumerate(STATE_NAMES):
    a = axes[i % 3][i // 3]
    a.hist(clean[name], bins=40)
    a.set_title(name)
plt.tight_layout(); plt.show()

## Preprocess (steady-state)
Select inputs and outputs, split into train/validation/test, then scale.

The split is done before scaling: the scaler is fit on the training set only, otherwise information about the validation and test data leaks into training. The split is 70% train / 15% validation / 15% test, a simple random split. This is valid here because each steady-state row is an independent sample (unlike the dynamic case, where rows within one trajectory are correlated).

In [ ]:
# ---------- preprocess ----------
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = clean[INPUT_NAMES].values     # inputs:  Q1, Q2, Q3
Y = clean[STATE_NAMES].values     # outputs: 9 states (y = x)

X_tr, X_tmp, Y_tr, Y_tmp = train_test_split(X, Y, test_size=0.30, random_state=42)
X_val, X_te, Y_val, Y_te = train_test_split(X_tmp, Y_tmp, test_size=0.50, random_state=42)

xsc = StandardScaler().fit(X_tr)   # fit ONLY on training data
ysc = StandardScaler().fit(Y_tr)
Xtr_s, Xval_s, Xte_s = xsc.transform(X_tr), xsc.transform(X_val), xsc.transform(X_te)
Ytr_s, Yval_s, Yte_s = ysc.transform(Y_tr), ysc.transform(Y_val), ysc.transform(Y_te)

print(f"Train / Val / Test sizes: {len(X_tr)} / {len(X_val)} / {len(X_te)}")

## Steady-state AI models: Linear / MLP / TSK-Fuzzy
Three models are trained and compared:
1. **Linear regression**: baseline, y = f(u) linear in u.
2. **MLP (nonlinear regression)**: 2 hidden layers × 32 neurons, ReLU, Adam optimizer, early stopping.
3. **TSK (Sugeno) fuzzy model**: 2 Gaussian membership functions per input (Low/High), first-order Sugeno consequents (linear in the inputs), membership centers placed at the 25th and 75th percentile of each input, and consequent parameters fitted by weighted least squares for each rule.

In [ ]:
# ---------- models ----------
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

# 1) Linear regression baseline
lin = LinearRegression().fit(Xtr_s, Ytr_s)

# 2) MLP (hidden layers, ReLU, Adam, epochs, batch size)
mlp = MLPRegressor(hidden_layer_sizes=(32, 32), activation="relu", solver="adam",
                    learning_rate_init=1e-3, batch_size=32, max_iter=2000,
                    early_stopping=True, n_iter_no_change=30, random_state=42)
mlp.fit(Xtr_s, Ytr_s)

# 3) TSK (Sugeno) fuzzy model, first-order, 2 MFs/input, WLS consequents
class TSKFuzzy:
    """First-order Sugeno model. n_mf Gaussian MFs per input; rules = all MF combinations
    (product firing strength); consequent = linear-in-inputs, fit by weighted least squares
    per rule (weights = normalized firing strength)."""
    def __init__(self, n_mf=2):
        self.n_mf = n_mf
    def _mfs(self, X):
        self.centers_, self.widths_ = [], []
        for j in range(X.shape[1]):
            lo, hi = np.percentile(X[:, j], 25), np.percentile(X[:, j], 75)
            self.centers_.append([lo, hi]); self.widths_.append(max((hi - lo) / 1.5, 1e-3))
    def _firing(self, X):
        d = X.shape[1]
        mf = np.zeros((X.shape[0], d, self.n_mf))
        for j in range(d):
            for m in range(self.n_mf):
                c, w = self.centers_[j][m], self.widths_[j]
                mf[:, j, m] = np.exp(-0.5 * ((X[:, j] - c) / w) ** 2)
        combos = np.array(np.meshgrid(*[range(self.n_mf)] * d)).T.reshape(-1, d)
        fire = np.ones((X.shape[0], len(combos)))
        for r, combo in enumerate(combos):
            for j, m in enumerate(combo): fire[:, r] *= mf[:, j, m]
        self.combos_ = combos
        return fire
    def fit(self, X, Y):
        self._mfs(X)
        fire = self._firing(X)
        norm = fire / (fire.sum(axis=1, keepdims=True) + 1e-12)
        Xa = np.hstack([X, np.ones((X.shape[0], 1))])
        self.params_ = np.zeros((fire.shape[1], Xa.shape[1], Y.shape[1]))
        for r in range(fire.shape[1]):
            Ww = np.sqrt(norm[:, r])[:, None]
            coef, *_ = np.linalg.lstsq(Xa * Ww, Y * Ww, rcond=None)
            self.params_[r] = coef
        return self
    def predict(self, X):
        fire = self._firing(X)
        norm = fire / (fire.sum(axis=1, keepdims=True) + 1e-12)
        Xa = np.hstack([X, np.ones((X.shape[0], 1))])
        out = np.zeros((X.shape[0], self.params_.shape[2]))
        for r in range(self.params_.shape[0]):
            out += norm[:, [r]] * (Xa @ self.params_[r])
        return out

tsk = TSKFuzzy(n_mf=2).fit(Xtr_s, Ytr_s)
models = {"Linear": lin, "MLP": mlp, "TSK-Fuzzy": tsk}
print("Models trained:", list(models.keys()))
print("MLP training iterations used:", mlp.n_iter_)

## Evaluate the steady-state models
Compute RMSE, MAE, R² on train/validation/test (in **original, unscaled units**, i.e. mass fractions and Kelvin) for all three models, then show per-output R² and actual-vs-predicted plots for the best model.

In [ ]:
# ---------- evaluate ----------
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate(model, Xs, Ys_scaled):
    pred = ysc.inverse_transform(model.predict(Xs))
    true = ysc.inverse_transform(Ys_scaled)
    rmse = np.sqrt(mean_squared_error(true, pred))
    mae = mean_absolute_error(true, pred)
    r2 = r2_score(true, pred)
    return rmse, mae, r2, pred, true

print("Steady-state model comparison (RMSE / MAE / R^2, all 9 outputs, original units)\n")
results = {}
for name, m in models.items():
    tr = evaluate(m, Xtr_s, Ytr_s); va = evaluate(m, Xval_s, Yval_s); te = evaluate(m, Xte_s, Yte_s)
    results[name] = dict(train=tr[:3], val=va[:3], test=te[:3])
    print(f"{name:10s} | train RMSE={tr[0]:.4f} MAE={tr[1]:.4f} R2={tr[2]:.4f}"
          f" | val RMSE={va[0]:.4f} MAE={va[1]:.4f} R2={va[2]:.4f}"
          f" | test RMSE={te[0]:.4f} MAE={te[1]:.4f} R2={te[2]:.4f}")

best_name = max(results, key=lambda k: results[k]["test"][2])
print(f"\nBest model by test R2: {best_name}")
best_model = models[best_name]
_, _, _, pred, true = evaluate(best_model, Xte_s, Yte_s)
per_out_r2 = pd.Series([r2_score(true[:, i], pred[:, i]) for i in range(9)], index=STATE_NAMES)
print("\nPer-output test R2 (best model):"); print(per_out_r2.round(4))

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i, name in enumerate(STATE_NAMES):
    a = axes[i % 3][i // 3]
    a.scatter(true[:, i], pred[:, i], s=8, alpha=0.5)
    lims = [min(true[:, i].min(), pred[:, i].min()), max(true[:, i].max(), pred[:, i].max())]
    a.plot(lims, lims, "k--", lw=1)
    a.set_title(f"{name}  (R2={per_out_r2[name]:.3f})"); a.set_xlabel("actual"); a.set_ylabel("predicted")
plt.suptitle(f"Actual vs Predicted — {best_name} (test set)"); plt.tight_layout(); plt.show()

# Residual plot for the best model (test set)
resid = true - pred
fig, axes = plt.subplots(3, 3, figsize=(12, 8))
for i, name in enumerate(STATE_NAMES):
    a = axes[i % 3][i // 3]
    a.scatter(pred[:, i], resid[:, i], s=8, alpha=0.5)
    a.axhline(0, color="k", lw=0.8)
    a.set_title(name); a.set_xlabel("predicted"); a.set_ylabel("residual")
plt.suptitle(f"Residuals — {best_name} (test set)"); plt.tight_layout(); plt.show()

---
# Part C — Dynamic model: user-defined input trajectories

- **User input** = the Q1/Q2/Q3 schedule and initial state defined in the input cell.
- **Simulated data** = the state trajectories x(t) that the first-principles ODEs produce from that input.
- **AI dataset** = the assembled (input, state) table that the dynamic AI model trains on.

The input cell ships with `INTERACTIVE = False` and a set of generated trajectories so the notebook runs top to bottom. Set `INTERACTIVE = True` and re-run it to enter your own schedules. The generated trajectories are random within the Q bounds and are not measured data.

## User input interface for dynamic inputs
A function that prompts for a trajectory's duration, sampling interval, a sequence of (time, Q1, Q2, Q3) schedule points and an initial state.

**Bounds enforced:** Q1, Q3 ∈ [0, 4.87×10⁶] kJ/h and Q2 ∈ [0, 1.68×10⁶] kJ/h.

**Dataset size:** the paper's closed-loop response settles in about 1.25 h (Fig. 3), so 1.7 h at dt = 0.01 h gives ~170 points per trajectory, comfortably past settling. `N_TRAJ = 12` gives roughly 2000 observations in total. Increase it for more variety in the dynamic dataset.

In [ ]:
# ---------- user input interface ----------
def collect_trajectory_interactive(traj_id):
    print(f"--- Trajectory {traj_id}: define Q1,Q2,Q3 schedule (kJ/h) ---")
    print(f"Bounds: Q1,Q3 in [{U_MIN[0]:.3e},{U_MAX[0]:.3e}], Q2 in [{U_MIN[1]:.3e},{U_MAX[1]:.3e}]")
    t_end = float(input("Trajectory duration [h] (paper settling ~1.25h, try 1.5-2h): "))
    dt_in = input(f"Sampling interval dt [h] (blank = paper default {DT_PAPER}): ").strip()
    dt = float(dt_in) if dt_in else DT_PAPER
    n_changes = int(input("Number of input segments (>=1): "))
    schedule = []
    for k in range(n_changes):
        t = float(input(f"  Segment {k+1} start time [h]: "))
        q1 = float(input("  Q1 [kJ/h]: ")); q2 = float(input("  Q2 [kJ/h]: ")); q3 = float(input("  Q3 [kJ/h]: "))
        schedule.append((t, q1, q2, q3))
    x0_in = input("Initial state, 9 comma-separated values [blank = paper's x_init]: ").strip()
    x0 = X_INIT_PAPER.copy() if not x0_in else np.array([float(v) for v in x0_in.split(",")])
    return dict(traj_id=traj_id, schedule=schedule, t_end=t_end, dt=dt, x0=x0)

# Randomly generated trajectories within the Q bounds.
# Set INTERACTIVE = True and re-run this cell to enter your own instead.
rng = np.random.default_rng(7)
N_TRAJ = 12
MANUAL_TRAJECTORIES = []
for i in range(N_TRAJ):
    q_start = U_MIN + rng.random(3) * (U_MAX - U_MIN)
    q_step  = U_MIN + rng.random(3) * (U_MAX - U_MIN)
    t_end = 1.7
    schedule = [(0.0, *q_start), (t_end / 2, *q_step)]   # one step change mid-trajectory
    MANUAL_TRAJECTORIES.append(dict(traj_id=i, schedule=schedule, t_end=t_end, dt=0.01,
                                     x0=X_INIT_PAPER.copy()))

INTERACTIVE = False   # <-- set True in Colab to enter your own trajectories
if INTERACTIVE:
    MANUAL_TRAJECTORIES = [collect_trajectory_interactive(i) for i in range(N_TRAJ)]

mode_label = "INTERACTIVE user input" if INTERACTIVE else "randomly generated trajectories"
print(f"{len(MANUAL_TRAJECTORIES)} trajectory schedules ready ({mode_label})")

## Convert schedules to u(t) and display before simulating
Each trajectory's (time, Q1, Q2, Q3) list is turned into a piecewise-constant function u(t), every step is checked against the bounds, and the schedule is plotted so the inputs can be inspected before the ODE solver runs.

In [ ]:
# ---------- build & display u(t) ----------
def schedule_to_ufunc(schedule):
    schedule = sorted(schedule, key=lambda r: r[0])
    times = np.array([r[0] for r in schedule])
    Qs = np.array([r[1:] for r in schedule])
    assert times[0] == 0.0, "schedule must start at t=0"
    for q in Qs:
        assert np.all(q >= U_MIN - 1e-6) and np.all(q <= U_MAX + 1e-6), f"Q out of bounds: {q}"
    def u(t):
        idx = max(np.searchsorted(times, t, side="right") - 1, 0)
        return Qs[idx]
    return u, times, Qs

for traj in MANUAL_TRAJECTORIES:
    u_func, times, Qs = schedule_to_ufunc(traj["schedule"])
    traj["u_func"] = u_func

print("Example -- trajectory 0 schedule:")
display(pd.DataFrame(MANUAL_TRAJECTORIES[0]["schedule"], columns=["t_start_h", "Q1", "Q2", "Q3"]))

fig, ax = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
tplot = np.linspace(0, MANUAL_TRAJECTORIES[0]["t_end"], 300)
for j, name in enumerate(INPUT_NAMES):
    vals = [MANUAL_TRAJECTORIES[0]["u_func"](t)[j] for t in tplot]
    ax[j].step(tplot, vals, where="post"); ax[j].set_ylabel(name)
ax[-1].set_xlabel("time [h]")
plt.suptitle("Trajectory 0: u(t) -- verify before simulating")
plt.tight_layout(); plt.show()

## Run the dynamic first-principles simulations
`dx/dt = f(x, u(t))` is integrated with `simulate` for every trajectory, using each trajectory's own initial state and duration. This step produces the simulated data: the input schedules only decide what is simulated, not the resulting x(t).

In [ ]:
# ---------- run dynamic simulations ----------
sim_results = []
for traj in MANUAL_TRAJECTORIES:
    res = simulate(traj["u_func"], traj["x0"], traj["t_end"], dt=traj["dt"])
    sim_results.append(res)
    print(f"trajectory {traj['traj_id']}: n_points={len(res['t'])} solver_ok={res['ok']} | {res['msg']}")

## Assemble the dynamic AI dataset
Long-format table: one row per (trajectory, time) with the inputs and the 9 states (= outputs, y = x). This is the **AI DATASET** that Part D (dynamic AI model) will train on.

In [ ]:
# ---------- assemble dynamic dataset ----------
rows = []
for traj, res in zip(MANUAL_TRAJECTORIES, sim_results):
    for k, t in enumerate(res["t"]):
        Q = traj["u_func"](t)
        row = dict(trajectory_id=traj["traj_id"], time=t, Q1=Q[0], Q2=Q[1], Q3=Q[2])
        for name, val in zip(STATE_NAMES, res["X"][k]): row[name] = val
        rows.append(row)
df_dyn = pd.DataFrame(rows)
n_traj = df_dyn["trajectory_id"].nunique()
print(f"Dynamic dataset shape: {df_dyn.shape}  ({n_traj} trajectories, ~{df_dyn.shape[0]//n_traj} points/trajectory)")
df_dyn.to_csv("dynamic_dataset.csv", index=False)
df_dyn.head()

## Validate the dynamic dataset
Check for NaN/solver failures, duplicate (trajectory, time) rows, non-physical states, and plot inputs/states across several trajectories to visually confirm the dynamics look sensible (transients, no discontinuities from numerical problems).

In [ ]:
# ---------- validate dynamic dataset ----------
n_nan = df_dyn[STATE_NAMES].isna().any(axis=1).sum()
n_dup = df_dyn.duplicated(subset=["trajectory_id", "time"]).sum()
bad_phys = (~df_dyn.apply(lambda r: is_physical(r[STATE_NAMES].values), axis=1)).sum()
failed_traj = [traj["traj_id"] for traj, res in zip(MANUAL_TRAJECTORIES, sim_results) if not res["ok"]]

print("NaN rows:", n_nan, "| duplicate (trajectory,time) rows:", n_dup,
      "| non-physical rows:", bad_phys, "| failed trajectories:", failed_traj)

fig, axes = plt.subplots(3, 3, figsize=(12, 8))
n_show = min(6, df_dyn["trajectory_id"].nunique())
for tid in df_dyn["trajectory_id"].unique()[:n_show]:
    sub = df_dyn[df_dyn["trajectory_id"] == tid]
    for i, name in enumerate(STATE_NAMES):
        a = axes[i % 3][i // 3]
        a.plot(sub["time"], sub[name], alpha=0.7, lw=1, label=f"traj {tid}" if i == 0 else None)
        a.set_title(name); a.set_xlabel("time [h]")
axes[0][0].legend(fontsize=6, ncol=2)
plt.suptitle(f"State trajectories, first {n_show} of {df_dyn['trajectory_id'].nunique()} trajectories")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for tid in df_dyn["trajectory_id"].unique()[:n_show]:
    sub = df_dyn[df_dyn["trajectory_id"] == tid]
    for j, name in enumerate(INPUT_NAMES):
        axes[j].step(sub["time"], sub[name], where="post", alpha=0.7, lw=1)
        axes[j].set_ylabel(name)
axes[-1].set_xlabel("time [h]")
plt.suptitle("Input trajectories u(t), same trajectories as above")
plt.tight_layout(); plt.show()

---
# Part D — Dynamic AI models: NARX Linear / MLP / TSK-Fuzzy

**NARX formulation:** y(k) = f(y(k-1), u(k), ...). The simplest lag order is used: regressor = [x(k), u(k)] (9 + 3 = 12 features), target = x(k+1). This matches a one-sample-ahead discretization of the ODE and keeps the TSK model tractable. Extending to [x(k), x(k-1), u(k), u(k-1)] is a straightforward way to explore longer memory.

## Preprocess the dynamic dataset (trajectory-level split + NARX construction)
**Why split by trajectory, not by row:** rows within one trajectory are strongly autocorrelated (x(k+1) is almost x(k) at small dt). A random row-level split would put near-duplicate points in both train and test, leaking information and making test accuracy look artificially good. Splitting whole trajectories (70% train / 15% validation / 15% test) keeps every test point unseen.

**Scaling:** fitted on the training NARX samples only, the same leakage rule as in the steady-state case.

In [ ]:
# ---------- preprocess dynamic dataset ----------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

traj_ids = df_dyn["trajectory_id"].unique()
tr_ids, tmp_ids = train_test_split(traj_ids, test_size=0.30, random_state=42)
val_ids, te_ids = train_test_split(tmp_ids, test_size=0.50, random_state=42)
print("Trajectory split -> train:", sorted(tr_ids), "| val:", sorted(val_ids), "| test:", sorted(te_ids))

def build_narx(df, ids):
    """regressor = [x(k), u(k)] (12), target = x(k+1) (9), built within each trajectory only."""
    Xs, Ys = [], []
    for tid in ids:
        sub = df[df["trajectory_id"] == tid].sort_values("time").reset_index(drop=True)
        Xk, Uk, Xk1 = sub[STATE_NAMES].values[:-1], sub[INPUT_NAMES].values[:-1], sub[STATE_NAMES].values[1:]
        Xs.append(np.hstack([Xk, Uk])); Ys.append(Xk1)
    return np.vstack(Xs), np.vstack(Ys)

Xtr_dyn, Ytr_dyn = build_narx(df_dyn, tr_ids)
Xval_dyn, Yval_dyn = build_narx(df_dyn, val_ids)
Xte_dyn, Yte_dyn = build_narx(df_dyn, te_ids)
print("NARX shapes (train/val/test):", Xtr_dyn.shape, Xval_dyn.shape, Xte_dyn.shape)

xsc_dyn = StandardScaler().fit(Xtr_dyn)
ysc_dyn = StandardScaler().fit(Ytr_dyn)
Xtr_dyn_s, Xval_dyn_s, Xte_dyn_s = xsc_dyn.transform(Xtr_dyn), xsc_dyn.transform(Xval_dyn), xsc_dyn.transform(Xte_dyn)
Ytr_dyn_s, Yval_dyn_s, Yte_dyn_s = ysc_dyn.transform(Ytr_dyn), ysc_dyn.transform(Yval_dyn), ysc_dyn.transform(Yte_dyn)

## Dynamic AI models: Linear / MLP / TSK-Fuzzy (NARX)
The same three-way comparison as in the steady-state case, now applied to the NARX regressor.

**TSK rule count:** partitioning all 12 NARX inputs with 2 membership functions each would need 2¹² = 4096 rules. `TSKFuzzyNARX` therefore partitions only the 3 current inputs u(k) into Low/High (2³ = 8 rules), while each rule's first-order Sugeno consequent is still linear in the full 12-dimensional regressor (states included).

In [ ]:
# ---------- dynamic AI models ----------
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
lin_dyn = LinearRegression().fit(Xtr_dyn_s, Ytr_dyn_s)

mlp_dyn = MLPRegressor(hidden_layer_sizes=(64, 64), activation="relu", solver="adam",
                        learning_rate_init=1e-3, batch_size=32, max_iter=3000,
                        early_stopping=True, n_iter_no_change=30, random_state=42)
mlp_dyn.fit(Xtr_dyn_s, Ytr_dyn_s)

class TSKFuzzyNARX:
    """TSK fuzzy partitioned ONLY on u(k) (cols 9,10,11 of the regressor; 2 MFs -> 8 rules)
    to keep the rule count manageable. Consequent is linear in the FULL
    12-dim regressor [x(k), u(k)] (first-order Sugeno)."""
    def __init__(self, n_mf=2, partition_cols=None):
        self.n_mf = n_mf; self.partition_cols = partition_cols
    def _mfs(self, Xp):
        self.centers_, self.widths_ = [], []
        for j in range(Xp.shape[1]):
            lo, hi = np.percentile(Xp[:, j], 25), np.percentile(Xp[:, j], 75)
            self.centers_.append([lo, hi]); self.widths_.append(max((hi - lo) / 1.5, 1e-3))
    def _firing(self, Xp):
        d = Xp.shape[1]
        mf = np.zeros((Xp.shape[0], d, self.n_mf))
        for j in range(d):
            for m in range(self.n_mf):
                c, w = self.centers_[j][m], self.widths_[j]
                mf[:, j, m] = np.exp(-0.5 * ((Xp[:, j] - c) / w) ** 2)
        combos = np.array(np.meshgrid(*[range(self.n_mf)] * d)).T.reshape(-1, d)
        fire = np.ones((Xp.shape[0], len(combos)))
        for r, combo in enumerate(combos):
            for j, m in enumerate(combo): fire[:, r] *= mf[:, j, m]
        return fire
    def fit(self, X, Y):
        Xp = X[:, self.partition_cols]
        self._mfs(Xp)
        fire = self._firing(Xp)
        norm = fire / (fire.sum(axis=1, keepdims=True) + 1e-12)
        Xa = np.hstack([X, np.ones((X.shape[0], 1))])
        self.params_ = np.zeros((fire.shape[1], Xa.shape[1], Y.shape[1]))
        for r in range(fire.shape[1]):
            Ww = np.sqrt(norm[:, r])[:, None]
            coef, *_ = np.linalg.lstsq(Xa * Ww, Y * Ww, rcond=None)
            self.params_[r] = coef
        return self
    def predict(self, X):
        Xp = X[:, self.partition_cols]
        fire = self._firing(Xp)
        norm = fire / (fire.sum(axis=1, keepdims=True) + 1e-12)
        Xa = np.hstack([X, np.ones((X.shape[0], 1))])
        out = np.zeros((X.shape[0], self.params_.shape[2]))
        for r in range(self.params_.shape[0]):
            out += norm[:, [r]] * (Xa @ self.params_[r])
        return out

tsk_dyn = TSKFuzzyNARX(n_mf=2, partition_cols=[9, 10, 11]).fit(Xtr_dyn_s, Ytr_dyn_s)
dyn_models = {"Linear": lin_dyn, "MLP": mlp_dyn, "TSK-Fuzzy": tsk_dyn}
print("Dynamic models trained:", list(dyn_models.keys()))

## Evaluate the dynamic models
Two distinct evaluations, because they measure different things:
1. **One-step-ahead accuracy** — how well the model predicts x(k+1) given the *true* x(k), u(k). At dt = 0.01 h the state barely changes per step, so near-perfect R² here is expected and not by itself proof of a good dynamic model.
2. **Multi-step (open-loop) rollout** — starting only from x(0), feed the model's own predictions back in as x(k) for 171 consecutive steps, using only the *true* input trajectory u(t). This is the real test of whether the model can simulate a whole trajectory, and is where prediction error compounds.

In [ ]:
# ---------- evaluate dynamic models ----------
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def eval_onestep(model, Xs, Ys_s):
    pred = ysc_dyn.inverse_transform(model.predict(Xs))
    true = ysc_dyn.inverse_transform(Ys_s)
    return np.sqrt(mean_squared_error(true, pred)), mean_absolute_error(true, pred), r2_score(true, pred)

print("One-step-ahead NARX comparison (RMSE / MAE / R2, original units)\n")
dyn_results = {}
for name, m in dyn_models.items():
    tr = eval_onestep(m, Xtr_dyn_s, Ytr_dyn_s); va = eval_onestep(m, Xval_dyn_s, Yval_dyn_s); te = eval_onestep(m, Xte_dyn_s, Yte_dyn_s)
    dyn_results[name] = dict(train=tr, val=va, test=te)
    print(f"{name:10s} | train RMSE={tr[0]:.4f} R2={tr[2]:.4f} | val RMSE={va[0]:.4f} R2={va[2]:.4f}"
          f" | test RMSE={te[0]:.4f} R2={te[2]:.4f}")

def rollout(model, x0, u_traj, n_steps):
    """Open-loop multi-step simulation: feed predictions back in as the next state."""
    X = np.zeros((n_steps + 1, 9)); X[0] = x0
    for k in range(n_steps):
        reg = np.hstack([X[k], u_traj[k]]).reshape(1, -1)
        pred_s = model.predict(xsc_dyn.transform(reg))
        X[k + 1] = ysc_dyn.inverse_transform(pred_s)[0]
    return X

best_dyn = max(dyn_results, key=lambda k: dyn_results[k]["test"][2])
print(f"\nBest dynamic model by one-step test R2: {best_dyn}")

test_tid = sorted(te_ids)[0]
sub = df_dyn[df_dyn["trajectory_id"] == test_tid].sort_values("time").reset_index(drop=True)
x0 = sub[STATE_NAMES].values[0]; u_traj = sub[INPUT_NAMES].values[:-1]
X_roll = rollout(dyn_models[best_dyn], x0, u_traj, len(u_traj))
true_traj = sub[STATE_NAMES].values
rmse_rollout = np.sqrt(mean_squared_error(true_traj, X_roll))
max_err = np.max(np.abs(true_traj - X_roll), axis=0)
print(f"Multi-step rollout RMSE on held-out trajectory {test_tid}: {rmse_rollout:.4f}")
print("Per-state max abs error over the rollout:")
print(pd.Series(max_err, index=STATE_NAMES).round(3))

fig, axes = plt.subplots(3, 3, figsize=(12, 8))
for i, name in enumerate(STATE_NAMES):
    a = axes[i % 3][i // 3]
    a.plot(sub["time"], true_traj[:, i], label="actual", lw=1.5)
    a.plot(sub["time"], X_roll[:, i], "--", label="predicted (rollout)", lw=1.2)
    a.set_title(name); a.set_xlabel("time [h]")
axes[0][0].legend(fontsize=7)
plt.suptitle(f"Multi-step rollout vs actual -- {best_dyn}, held-out trajectory {test_tid}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
err_traj = true_traj - X_roll
for j, name in enumerate(["xA1", "T1", "xA3"]):
    idx = STATE_NAMES.index(name)
    axes[j].plot(sub["time"], err_traj[:, idx]); axes[j].axhline(0, color="k", lw=0.6)
    axes[j].set_ylabel(f"error: {name}")
axes[-1].set_xlabel("time [h]")
plt.suptitle("Rollout prediction error vs time (sample states)")
plt.tight_layout(); plt.show()

print("\nNote: one-step R2 is near 1.0 for all three models because dt=0.01h is small relative to the\n"
      "process time constants, so x(k+1) is close to x(k) -- a near-identity mapping is easy to fit.\n"
      "The multi-step rollout is the harder, more meaningful test: prediction error compounds over\n"
      "171 steps because small one-step errors feed back into the next prediction. This is a genuine\n"
      "limitation of a purely data-driven one-step NARX model trained on only 8 trajectories, and is\n"
      "worth noting as a limitation (more trajectories, a longer lag order, or a proper multi-step\n"
      "training loss would likely reduce it -- this last point is exactly the motivation for the\n"
      "input-augmented Koopman model of the reference paper).")

## Save datasets and trained models
Persist the fitted models and scalers (needed to reproduce predictions later, e.g. in Stage 2/3) and confirm both CSV datasets are on disk.

In [ ]:
# ---------- save datasets and models ----------
import joblib

joblib.dump({"models": dyn_models, "xsc": xsc_dyn, "ysc": ysc_dyn}, "dynamic_models.joblib")
joblib.dump({"models": models, "xsc": xsc, "ysc": ysc}, "steadystate_models.joblib")

import os
for f in ["steady_state_dataset.csv", "dynamic_dataset.csv", "dynamic_models.joblib", "steadystate_models.joblib"]:
    path = f"{f}"
    status = f"OK, {os.path.getsize(path)} bytes" if os.path.exists(path) else "MISSING"
    print(f"{f:30s} {status}")

---
# Final Stage 1 summary

| Item | Result |
|---|---|
| **Process** | Reactor–separator: 2 CSTRs + flash separator, reactions A→B→C |
| **First-principles model** | 9 ODEs + flash algebraic equations; rate constants (×3600 unit fix) and energy-balance scale factors (C_RXN=0.004, C_VAP=2.0) calibrated against the published steady state |
| **Inputs** | u = [Q1, Q2, Q3] (jacket heat duties, kJ/h) |
| **Outputs / states** | y = x = 9 states (xA, xB, T per vessel) |
| **Steady-state dataset** | 2000 LHS samples, all converged; narrowed input lower bound |
| **Dynamic dataset** | 12 trajectories × ~171 pts = 2052 rows; randomly generated input schedules (editable via the input cell) |
| **Preprocessing** | Steady-state: random 70/15/15 split (independent samples). Dynamic: trajectory-level 70/15/15 split (avoids leakage from autocorrelated rows) |
| **Steady-state AI models** | Linear / MLP / TSK-Fuzzy |
| **Dynamic AI models** | Same three, NARX formulation: regressor [x(k),u(k)] -> x(k+1) |
| **Steady-state test R²** | Linear 0.671, **MLP 0.988**, TSK-Fuzzy 0.720 |
| **Dynamic one-step test R²** | all ≈ 0.94-1.00 (misleadingly high at small dt) |
| **Dynamic multi-step rollout** | RMSE ≈ 12.3 (mixed units) on a held-out trajectory, the real dynamic-accuracy number |
| **Limitations** | (1) the dynamic trajectories are randomly generated, not measured plant data; (2) the TSK model's fuzzy partition is restricted to 3 of 12 NARX inputs to stay tractable; (3) open-loop rollout error is substantial, a known weakness of one-step NARX training that the Koopman approach targets |

This completes Stage 1 — Modeling.

---
# Part E — Stage 2: Optimization

The Stage 1 steady-state MLP (best test R² 0.988) is used as a surrogate inside a Genetic Algorithm to find optimal heating duties, rather than treating optimization as a separate exercise.

- **Method:** a from-scratch Genetic Algorithm (tournament selection, arithmetic crossover, Gaussian mutation, elitism).
- **Objective:** the B production rate leaving through the purge stream, **Ḃ = ρ · Fp · x_Br(x_A3, x_B3)** [kg/h], computed from the model's own parameters (Fp = 0.504 m³/h) and relative-volatility relations.
- **Case A:** maximize Ḃ within the input bounds.
- **Case B:** minimize total heating duty (Q1+Q2+Q3) subject to Ḃ ≥ 90% of Case A's optimum.

Every GA result is validated against the true first-principles steady-state solver, to check that the surrogate's optimum is also a good operating point for the real process.

## Define the optimization problem
**Decision variables:** u = [Q1, Q2, Q3] (same 3 inputs as Stage 1), bounded by the paper's own limits (`U_MIN`, `U_MAX`).
**Predictive model:** the Stage 1 MLP (`models["MLP"]`, `xsc`, `ysc` — already fitted in the steady-state section, reused as-is here, not retrained).
**Objective (Case A):** maximize Ḃ = ρ·Fp·x_Br. A soft penalty keeps the GA away from input regions where the MLP's own predictions leave the physically valid range (mass fractions outside [0,1], T below 250 K) — the surrogate is not reliable there (Stage 1 Cell 6 only validated it inside the sampled training range).

In [ ]:
# ---------- define the optimization problem ----------
def x_Br_from_x3(xA3, xB3):
    """Purge/overhead B mass fraction from the separator liquid composition,
    using the SAME relative-volatility relations as the first-principles model."""
    xC3 = 1.0 - xA3 - xB3
    den = alpha[0]*xA3 + alpha[1]*xB3 + alpha[2]*xC3
    return alpha[1]*xB3/den

def surrogate_predict(U):
    """U: (n,3) array of [Q1,Q2,Q3] -> (n,9) predicted states, original units, via Stage 1 MLP."""
    U = np.atleast_2d(U)
    Ys = models["MLP"].predict(xsc.transform(U))
    return ysc.inverse_transform(Ys)

def physical_penalty(X):
    """Soft penalty: 0 inside the physically valid / trained region, grows outside it."""
    xa, xb, T = X[:, [0, 3, 6]], X[:, [1, 4, 7]], X[:, [2, 5, 8]]
    return (np.clip(-xa, 0, None).sum(1) + np.clip(xa - 1, 0, None).sum(1) +
            np.clip(-xb, 0, None).sum(1) + np.clip(xb - 1, 0, None).sum(1) +
            np.clip(xa + xb - 1, 0, None).sum(1) + np.clip(250 - T, 0, None).sum(1))

def neg_Bdot_caseA(U):
    """Case A fitness (GA MINIMIZES): -Bdot + penalty."""
    X = surrogate_predict(U)
    Bdot = rho * Fp * x_Br_from_x3(X[:, 6], X[:, 7])
    return -Bdot + 1e6 * physical_penalty(X)

bounds = np.column_stack([U_MIN, U_MAX])
baseline_Bdot = rho * Fp * x_Br_from_x3(*surrogate_predict(U_S.reshape(1, -1))[0, [6, 7]])
print(f"Decision variables: u = [Q1,Q2,Q3], bounds (kJ/h): {list(zip(U_MIN, U_MAX))}")
print(f"Baseline B production rate at paper's own steady-state input u_s: {baseline_Bdot:.2f} kg/h")

## Genetic Algorithm (course method)
Standard GA loop: initialize a random population within bounds, evaluate fitness, keep the best `elite_frac` unchanged (elitism), fill the rest via **tournament selection** + **arithmetic (blend) crossover** + **Gaussian mutation**, repeat for `n_gen` generations. Fitness is **minimized**, so a maximization objective (Case A) is negated.

In [ ]:
# ---------- genetic algorithm ----------
def genetic_algorithm(fitness_fn, bounds, pop_size=60, n_gen=80, cx_rate=0.8,
                       mut_rate=0.2, mut_scale=0.1, elite_frac=0.1, seed=0):
    rng = np.random.default_rng(seed)
    lo, hi = bounds[:, 0], bounds[:, 1]
    d = len(lo)
    pop = lo + rng.random((pop_size, d)) * (hi - lo)
    n_elite = max(1, int(elite_frac * pop_size))
    history = []
    for gen in range(n_gen):
        fit = fitness_fn(pop)
        order = np.argsort(fit)
        pop, fit = pop[order], fit[order]
        history.append(fit[0])
        elites = pop[:n_elite].copy()
        def tournament(k=3):
            idx = rng.integers(0, pop_size, size=k)
            return pop[idx[np.argmin(fit[idx])]]
        children = []
        while len(children) < pop_size - n_elite:
            p1, p2 = tournament(), tournament()
            if rng.random() < cx_rate:
                a = rng.random(d)
                child = a * p1 + (1 - a) * p2          # arithmetic (blend) crossover
            else:
                child = p1.copy()
            if rng.random() < mut_rate:
                child = child + rng.normal(0, mut_scale * (hi - lo))  # Gaussian mutation
            children.append(np.clip(child, lo, hi))
        pop = np.vstack([elites, np.array(children)])
    fit = fitness_fn(pop)
    best_idx = np.argmin(fit)
    return pop[best_idx], fit[best_idx], history

print("GA defined: tournament selection, arithmetic crossover, Gaussian mutation, elitism.")

## Case A: maximize B production rate
Run the GA, plot convergence (best fitness per generation — required to show the search actually converged, not just report a final number), then **validate the GA's optimum against the true first-principles model** (not just the surrogate it was optimized on).

In [ ]:
# ---------- Case A -- maximize B production rate ----------
best_u_A, best_fit_A, hist_A = genetic_algorithm(neg_Bdot_caseA, bounds, pop_size=60, n_gen=80, seed=1)
Bdot_A_surrogate = -best_fit_A

print("CASE A -- maximize B purge production rate, bounds-only")
print(f"  Optimal u* = Q1={best_u_A[0]:.3e}, Q2={best_u_A[1]:.3e}, Q3={best_u_A[2]:.3e} kJ/h")
print(f"  Surrogate-predicted B rate at u*: {Bdot_A_surrogate:.2f} kg/h  (vs {baseline_Bdot:.2f} kg/h at paper's u_s)")

plt.figure(figsize=(5, 3))
plt.plot(hist_A); plt.xlabel("generation"); plt.ylabel("best fitness (minimized)")
plt.title("GA convergence -- Case A"); plt.tight_layout(); plt.show()

# ---- validate against the TRUE first-principles model (steady_state solver) ----
xs_A_true, info_A = steady_state(best_u_A, x0=XS_REF_4DP)
ok_A, resid_A = info_A["ok"], info_A["resid"]
print(f"\nFirst-principles validation at u*: converged = {ok_A}, residual = {resid_A:.2e}")
if xs_A_true is not None:
    Bdot_A_true = rho * Fp * x_Br_from_x3(xs_A_true[6], xs_A_true[7])
    gap_pct = 100 * (Bdot_A_surrogate - Bdot_A_true) / Bdot_A_true
    print(f"True (first-principles) B rate at u*: {Bdot_A_true:.2f} kg/h")
    print(f"Surrogate vs true gap at the optimum: {gap_pct:+.1f}%  <- surrogate accuracy AT the optimum, not just on random test data")
    Bdot_base_true = rho * Fp * x_Br_from_x3(XS_REF_4DP[6], XS_REF_4DP[7])
    print(f"True B rate at the paper's own steady state (u_s): {Bdot_base_true:.2f} kg/h -> true change at u*: {Bdot_A_true - Bdot_base_true:+.2f} kg/h")
    Xpred_A = surrogate_predict(best_u_A)[0]
    cmp_A = pd.DataFrame({"state": STATE_NAMES, "surrogate": Xpred_A, "true": xs_A_true,
                          "abs_err": np.abs(Xpred_A - xs_A_true)})
    print(cmp_A.round(4))

## Case B: minimize energy subject to a production constraint
A constrained problem: **minimize Q1+Q2+Q3** subject to **Ḃ ≥ 90% of Case A's optimum**. Handled with a penalty method (standard for GA, since GA has no native constraint handling): the fitness adds a heavily weighted penalty for any shortfall below the target, normalized so the penalty dominates the energy term whenever the constraint is violated — this needed tuning (an un-normalized penalty let the GA ignore the constraint entirely in an earlier version of this cell).

In [ ]:
# ---------- Case B -- minimize energy subject to Bdot >= 90% of Case A ----------
target_frac = 0.90
target_Bdot = target_frac * Bdot_A_surrogate

def energy_caseB(U):
    U = np.atleast_2d(U)
    X = surrogate_predict(U)
    Bdot = rho * Fp * x_Br_from_x3(X[:, 6], X[:, 7])
    energy_norm = U.sum(axis=1) / U_MAX.sum()                            # dimensionless, ~0-1
    viol_norm = np.clip((target_Bdot - Bdot) / target_Bdot, 0, None)     # fractional shortfall, 0-1
    return energy_norm + 20 * viol_norm + 1e3 * physical_penalty(X)

best_u_B, best_fit_B, hist_B = genetic_algorithm(energy_caseB, bounds, pop_size=60, n_gen=80, seed=2)
X_B = surrogate_predict(best_u_B)[0]
Bdot_B_surrogate = rho * Fp * x_Br_from_x3(X_B[6], X_B[7])

print("CASE B -- minimize total Q, subject to B rate >= 90% of Case A's optimum")
print(f"  Target B rate: {target_Bdot:.2f} kg/h | Achieved (surrogate): {Bdot_B_surrogate:.2f} kg/h")
print(f"  Optimal u* = Q1={best_u_B[0]:.3e}, Q2={best_u_B[1]:.3e}, Q3={best_u_B[2]:.3e} kJ/h")
print(f"  Total Q: {best_u_B.sum():.3e} kJ/h  (Case A total Q was {best_u_A.sum():.3e} kJ/h, "
      f"a {100*(1-best_u_B.sum()/best_u_A.sum()):.1f}% reduction)")

plt.figure(figsize=(5, 3))
plt.plot(hist_B); plt.xlabel("generation"); plt.ylabel("best fitness (minimized)")
plt.title("GA convergence -- Case B"); plt.tight_layout(); plt.show()

xs_B_true, info_B = steady_state(best_u_B, x0=XS_REF_4DP)
ok_B = info_B["ok"]
print(f"\nFirst-principles validation at u*: converged = {ok_B}")
if xs_B_true is not None:
    Bdot_B_true = rho * Fp * x_Br_from_x3(xs_B_true[6], xs_B_true[7])
    print(f"True B rate at u*: {Bdot_B_true:.2f} kg/h  (target was {target_Bdot:.2f} kg/h)")
    shortfall = 100 * (target_Bdot - Bdot_B_true) / target_Bdot
print(f"True-model shortfall vs target: {shortfall:.1f}%  ->", "constraint met" if shortfall <= 0 else "constraint met only within a 2% tolerance" if shortfall <= 2 else "constraint violated (surrogate gap)")

## GA vs. random search (efficiency check)
The GA is compared with a random-search baseline that uses the identical function-evaluation budget (`pop_size × n_gen`), to show whether the selection, crossover and mutation steps actually help compared with blind sampling.

In [ ]:
# ---------- GA vs random search, same evaluation budget ----------
n_evals = 60 * 80   # same budget as the GA runs above
rng = np.random.default_rng(3)
U_random = bounds[:, 0] + rng.random((n_evals, 3)) * (bounds[:, 1] - bounds[:, 0])
fit_random = neg_Bdot_caseA(U_random)
best_random = -fit_random.min()

print(f"Case A objective (B rate, kg/h) -- same {n_evals} function evaluations:")
print(f"  Genetic Algorithm : {Bdot_A_surrogate:.2f} kg/h")
print(f"  Random search     : {best_random:.2f} kg/h")
print(f"  GA improvement over random search: {100*(Bdot_A_surrogate-best_random)/best_random:+.1f}%")

# running-best curves on the same x-axis: one point per 60 evaluations (= one GA generation)
cum_best = -pd.Series(fit_random).cummin().values
rand_curve = cum_best[np.arange(1, 81) * 60 - 1]
plt.figure(figsize=(6, 3.5))
plt.plot(-np.array(hist_A), label="GA (best per generation)")
plt.plot(rand_curve, label="Random search (running best)")
plt.ylim(0.9 * Bdot_A_surrogate, 1.02 * max(rand_curve.max(), Bdot_A_surrogate))
plt.xlabel("generation (GA) / equivalent 60-evaluation batch (random)"); plt.ylabel("best B rate so far (kg/h)")
plt.legend(); plt.title("GA vs random search, same evaluation budget"); plt.tight_layout(); plt.show()

---
# Part F — Stage 2 summary

| Item | Result |
|---|---|
| **Surrogate model** | Stage 1 steady-state MLP, R²=0.988 on held-out test data |
| **Optimization method** | Genetic Algorithm, implemented from scratch |
| **Objective** | Ḃ = ρ·Fp·x_Br, the B production rate via the purge stream |
| **Case A** | Surrogate predicts 354 kg/h (vs 338 at the baseline input), but the true first-principles B rate at the optimum is 325 kg/h, **below** the true rate at the baseline (≈328 kg/h). The surrogate error (~9%) exceeds the predicted gain, so Case A shows no real improvement |
| **Case B** | Minimize total Q subject to Ḃ ≥ 90% of Case A's optimum: about 11.6% less total duty than Case A, with the true B rate within 2% of the target |
| **GA vs random search** | Same evaluation budget; random search matched the GA on this small problem |
| **Limitations** | (1) the objective is a modeling choice; a different objective (conversion, an economic criterion) would change the optimum; (2) the GA optimizes the surrogate, so the first-principles validation is what certifies a result; (3) GA hyperparameters use standard defaults and were not tuned |

---
# Part G — Stage 3: Control

The Stage 1 dynamic model is used to design an AI-based control strategy.

- **Controller:** Model Predictive Control (MPC) whose internal predictor is the Stage 1 dynamic NARX model (the TSK-Fuzzy model chosen in the dynamic evaluation), so the AI model is the control model.
- **MPC solver:** the Genetic Algorithm from Stage 2 (same operators), warm-started with the previous optimal input sequence.
- **Controlled variables:** T1, T2, T3; manipulated variables: Q1, Q2, Q3. The full state is measured.
- **Horizons:** control interval 0.01 h (the dynamic model's sample time), Np = 10, Nc = 3.
- **Input bounds:** the range of Q seen in the dynamic training data, so the model never extrapolates.
- **Plant:** the first-principles ODEs, so closed-loop results show how the controller copes with the AI model's real prediction error.
- **Offset-free correction:** each step the MPC adds the last observed model-vs-plant temperature mismatch to its predictions.
- **Baseline:** a decentralized PI controller (T1→Q1, T2→Q2, T3→Q3), with gains chosen as the best of a 6-point grid by IAE on each scenario.

**Scenarios:** (1) setpoint step to a reachable operating point (computed with the true steady-state solver); (2) disturbance rejection, a 15 K drop in fresh-feed temperature T10 at t = 0.05 h with setpoints held.

## MPC setup: predictor, plant, cost
`dyn_step` batches the Stage 1 NARX model over a whole GA population; `mpc_cost` rolls each candidate input sequence forward Np steps and sums the squared (scaled) temperature error plus a penalty on input moves (which also damps GA randomness).

In [ ]:
# ---------- MPC setup ----------
import time
Ts_ctrl = 0.01          # h, control interval = the dynamic model's training sample time
Np, Nc = 10, 3          # prediction / control horizons (steps)
dyn_model = dyn_models[best_dyn]
dyn_in_lo = df_dyn[INPUT_NAMES].min().values      # input range the dynamic model was trained on
dyn_in_hi = df_dyn[INPUT_NAMES].max().values
CV_IDX = [2, 5, 8]      # controlled variables: T1, T2, T3
print(f"Predictor: {best_dyn} dynamic NARX model (Stage 1) | Ts={Ts_ctrl} h, Np={Np}, Nc={Nc}")
print("MPC input bounds (kJ/h) = range seen in dynamic training data:", dyn_in_lo.round(-3), dyn_in_hi.round(-3))

def dyn_step(Xb, Ub):
    """Batch one-step prediction x(k+1)=f(x(k),u(k)); Xb (P,9), Ub (P,3)."""
    reg = np.hstack([Xb, Ub])
    return ysc_dyn.inverse_transform(dyn_model.predict(xsc_dyn.transform(reg)))

def plant_step(x, u, Ts=Ts_ctrl):
    """True process (first-principles ODEs) advanced one control interval."""
    from scipy.integrate import solve_ivp
    sol = solve_ivp(rhs, [0, Ts], x, args=(u,), method="LSODA", rtol=1e-8, atol=1e-10)
    return sol.y[:, -1]

T_scale = 10.0          # K, error scale
u_range = dyn_in_hi - dyn_in_lo
W_DU = 0.5              # weight on normalized input moves (damps GA jitter)

def mpc_cost(Z, x_meas, u_prev, T_sp, d_hat):
    """Z: (P, 3*Nc) candidate input sequences (GA population). Lower is better."""
    P = Z.shape[0]
    U_seq = Z.reshape(P, Nc, 3)
    Xp = np.repeat(x_meas.reshape(1, -1), P, axis=0)
    J = np.zeros(P)
    for k in range(Np):
        Uk = U_seq[:, min(k, Nc - 1), :]
        Xp = dyn_step(Xp, Uk)
        J += (((Xp[:, CV_IDX] + d_hat - T_sp) / T_scale) ** 2).sum(axis=1)   # d_hat: output-bias correction
    du = np.diff(np.concatenate([np.repeat(u_prev.reshape(1, 1, 3), P, axis=0), U_seq], axis=1), axis=1)
    J += W_DU * ((du / u_range) ** 2).sum(axis=(1, 2)) * 10
    return J

T10_nominal = float(T10)
def set_feed_T(val):
    """Disturbance hook: change the CSTR-1 fresh-feed temperature seen by the true plant (global T10)."""
    global T10
    T10 = float(val)

## GA-based MPC controller
At every control step the GA searches the 9 decision variables (3 inputs × Nc moves), applies only the first move, shifts the sequence, and repeats (receding horizon).

In [ ]:
# ---------- GA-based MPC controller ----------
def ga_mpc_solve(cost_fn, lo, hi, init_guess, pop_size=40, n_gen=25, mut_scale=0.05, elite_frac=0.1, seed=0):
    """Same GA operators as the earlier GA, warm-started: the previous optimal sequence (shifted) seeds the population."""
    rng = np.random.default_rng(seed)
    d = len(lo)
    pop = init_guess + rng.normal(0, 0.05, (pop_size, d)) * (hi - lo)
    pop[0] = init_guess
    pop[1:pop_size // 4] = lo + rng.random((pop_size // 4 - 1, d)) * (hi - lo)   # some global exploration
    pop = np.clip(pop, lo, hi)
    n_elite = max(1, int(elite_frac * pop_size))
    for gen in range(n_gen):
        fit = cost_fn(pop)
        order = np.argsort(fit); pop, fit = pop[order], fit[order]
        elites = pop[:n_elite].copy()
        children = []
        while len(children) < pop_size - n_elite:
            i = rng.integers(0, pop_size, 3); p1 = pop[i[np.argmin(fit[i])]]
            i = rng.integers(0, pop_size, 3); p2 = pop[i[np.argmin(fit[i])]]
            a = rng.random(d) if rng.random() < 0.8 else np.ones(d)
            c = a * p1 + (1 - a) * p2
            if rng.random() < 0.3:
                c = c + rng.normal(0, mut_scale * (hi - lo))
            children.append(np.clip(c, lo, hi))
        pop = np.vstack([elites, np.array(children)])
    fit = cost_fn(pop)
    return pop[np.argmin(fit)]

def run_mpc(x0, T_sp_fn, n_steps, u0, dist_fn=None):
    x, u_prev = x0.copy(), u0.copy()
    lo = np.tile(dyn_in_lo, Nc); hi = np.tile(dyn_in_hi, Nc)
    guess = np.tile(u0, Nc)
    Xh, Uh, solve_t = [x.copy()], [], []
    d_hat = np.zeros(3)      # offset-free feedback: model-vs-plant mismatch on the temperatures
    for k in range(n_steps):
        T_sp = T_sp_fn(k * Ts_ctrl)
        t0 = time.time()
        z = ga_mpc_solve(lambda Z: mpc_cost(Z, x, u_prev, T_sp, d_hat), lo, hi, guess, seed=k)
        solve_t.append(time.time() - t0)
        u = z[:3]; u_prev = u
        guess = np.concatenate([z[3:], z[-3:]])        # shift the sequence one step
        if dist_fn is not None: set_feed_T(dist_fn(k * Ts_ctrl))
        x_pred = dyn_step(x.reshape(1, -1), u.reshape(1, -1))[0]
        x = plant_step(x, u)
        d_hat = x[CV_IDX] - x_pred[CV_IDX]      # measured minus predicted, applied over the next horizon
        Xh.append(x.copy()); Uh.append(u.copy())
    set_feed_T(T10_nominal)
    return np.array(Xh), np.array(Uh), np.array(solve_t)

## PI baseline controller
Conventional decentralized PI with the same input bounds, for comparison.

In [ ]:
# ---------- PI baseline ----------
def run_pi(x0, T_sp_fn, n_steps, u0, Kc, tauI, dist_fn=None):
    """Decentralized PI: T_i -> Q_i (velocity form, clipped to the same input bounds as the MPC)."""
    x, u = x0.copy(), u0.copy()
    I = np.zeros(3)
    Xh, Uh = [x.copy()], []
    for k in range(n_steps):
        e = T_sp_fn(k * Ts_ctrl) - x[CV_IDX]
        I += e * Ts_ctrl
        u = np.clip(u0 + Kc * (e + I / tauI), dyn_in_lo, dyn_in_hi)
        if dist_fn is not None: set_feed_T(dist_fn(k * Ts_ctrl))
        x = plant_step(x, u)
        Xh.append(x.copy()); Uh.append(u.copy())
    set_feed_T(T10_nominal)
    return np.array(Xh), np.array(Uh)

## Closed-loop scenarios and comparison
IAE = integral of absolute temperature error [K·h] (lower is better). Stochastic GA is seeded per step, so results are reproducible.

In [ ]:
# ---------- closed-loop scenarios ----------
xs0, _ = steady_state(U_S, x0=XS_REF_4DP)
T_sp0 = xs0[CV_IDX]
n_steps, t_ax = 60, np.arange(61) * Ts_ctrl

def iae(X, sp_fn):
    sp = np.array([sp_fn(t) for t in t_ax])
    return np.abs(X[:, CV_IDX] - sp).sum(axis=0) * Ts_ctrl

# ---- Scenario 1: setpoint step to a reachable operating point (found with the true steady-state solver) ----
u_target = U_S * np.array([1.15, 1.25, 0.85])
xs_t, _ = steady_state(u_target, x0=XS_REF_4DP)
T_sp1 = xs_t[CV_IDX]
sp1 = lambda t: T_sp0 if t < 0.05 else T_sp1

# ---- Scenario 2: disturbance rejection, fresh-feed temperature T10 drops 15 K at t = 0.05 h, setpoints held ----
sp2 = lambda t: T_sp0
dist = lambda t: T10_nominal if t < 0.05 else T10_nominal - 15.0

def tune_pi(sp_fn, dist_fn=None):
    best = None
    for Kc in [1e5, 4e5, 1e6]:
        for tauI in [0.05, 0.2]:
            Xp, Up = run_pi(xs0, sp_fn, n_steps, U_S, Kc, tauI, dist_fn)
            score = iae(Xp, sp_fn).sum()
            if best is None or score < best[0]: best = (score, Kc, tauI, Xp, Up)
    return best

results = {}
for name, sp_fn, dist_fn in [("Scenario 1: setpoint step", sp1, None), ("Scenario 2: feed-temperature disturbance", sp2, dist)]:
    Xm, Um, st = run_mpc(xs0, sp_fn, n_steps, U_S, dist_fn)
    score, Kc, tauI, Xp, Up = tune_pi(sp_fn, dist_fn)
    results[name] = dict(Xm=Xm, Um=Um, Xp=Xp, Up=Up, sp_fn=sp_fn)
    print(f"{name}")
    print(f"  AI-MPC (GA + dynamic NARX) IAE [K*h] T1,T2,T3 = {iae(Xm, sp_fn).round(3)} | mean GA solve time {st.mean():.2f} s/step")
    print(f"  PI baseline (Kc={Kc:.0e}, tauI={tauI}, best of a 6-point grid) IAE = {iae(Xp, sp_fn).round(3)}")
    print(f"  final |T error| MPC = {np.abs(Xm[-1, CV_IDX] - sp_fn(1.0)).round(3)} K | PI = {np.abs(Xp[-1, CV_IDX] - sp_fn(1.0)).round(3)} K")

for name, r in results.items():
    fig, axes = plt.subplots(2, 3, figsize=(13, 6))
    for j in range(3):
        sp = np.array([r["sp_fn"](t)[j] for t in t_ax])
        axes[0][j].plot(t_ax, sp, "k:", label="setpoint")
        axes[0][j].plot(t_ax, r["Xm"][:, CV_IDX[j]], label="AI-MPC")
        axes[0][j].plot(t_ax, r["Xp"][:, CV_IDX[j]], "--", label="PI")
        axes[0][j].set_title(f"T{j+1} [K]")
        axes[1][j].step(t_ax[:-1], r["Um"][:, j], where="post", label="AI-MPC")
        axes[1][j].step(t_ax[:-1], r["Up"][:, j], where="post", ls="--", label="PI")
        axes[1][j].set_title(f"Q{j+1} [kJ/h]"); axes[1][j].set_xlabel("time [h]")
    axes[0][0].legend(fontsize=7)
    plt.suptitle(name); plt.tight_layout(); plt.show()

---
# Part H — Stage 3 summary

| Item | Result |
|---|---|
| **AI model used** | Stage 1 dynamic NARX TSK-Fuzzy model as the MPC predictor (one-step test R² ≈ 1.00; multi-step open-loop drift is its weakness, mitigated because MPC restarts from the measured state every step) |
| **Optimizer** | Genetic Algorithm, warm-started, ~0.2 s per control step |
| **Setpoint step** | AI-MPC had lower IAE than the tuned PI on all three temperatures (≈0.47/0.56/0.40 vs 0.55/0.63/0.75 K·h) |
| **Disturbance rejection** | **PI was clearly better** (IAE ≈ 0.04/0.01/0.00 vs ≈0.31/0.35/0.15 K·h for MPC, which left ≈0.3–0.5 K residual offset) |
| **Limitations** | (1) disturbance handling relies on a one-step output-bias estimate; the model has no disturbance input, so a disturbance model or integral action would be the fix; (2) the GA gives approximate per-step solutions and is slower than a gradient solver (not benchmarked); (3) horizons, weights, controlled-variable choice and the PI grid are design choices, not tuned systematically; (4) full-state measurement and no noise; (5) only two scenarios and one seed, so the AI-MPC-vs-PI ranking is indicative, not conclusive |

This completes Stages 1–3.